# SOFC Voltage Forecasting — Colab Runner (LSTM, Seq2Seq LSTM, TCN)

Notebook này train **LSTM, Seq2Seq LSTM, TCN** cho target **`V`** (điện áp SOFC) trên Colab với GPU miễn phí. Random Forest và XGBoost chạy riêng ở local (không cần GPU, xem `src/main.py`).

**Cách hoạt động:** vì `E:\sofc` chưa phải git repo, ta KHÔNG clone/pull như project FCF cũ -- thay vào đó upload thẳng 2 thứ lên Google Drive:
1. Toàn bộ thư mục `src/` (gồm `models/` bên trong: `random_forest.py`, `xgboost_model.py`, `lstm.py`, `seq2seq_lstm.py`, `tcn.py`)
2. File data thô `data/raw/DataTime_export.csv`

Notebook chỉ `cd` vào `src/` trên Drive rồi chạy thẳng `python main_lstm.py` / `main_tcn.py` / `main_seq2seq.py` -- giống hệt cách FCF cũ chạy (`!python main_lstm_power.py`...). Mỗi script tự lo phần load data, window, train, evaluate, lưu kết quả -- notebook không nhúng logic training, chỉ gọi script.

**Trước khi chạy:**
1. `Runtime -> Change runtime type -> GPU (T4)`.
2. Trên Google Drive (MyDrive), tạo thư mục `sofc/`, upload `src/` (nguyên thư mục, gồm cả `src/models/`) và `data/raw/DataTime_export.csv` vào đó, giữ đúng cấu trúc `sofc/data/raw/DataTime_export.csv`.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Cấu hình đường dẫn project trên Drive

In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/sofc"
SRC_DIR = f"{DRIVE_PROJECT_DIR}/src"


## 3. Kiểm tra file cần thiết đã có trên Drive chưa

Nếu thiếu file nào, upload từ máy local (đường dẫn tương ứng trong `E:\sofc\`) vào đúng vị trí trên Drive rồi chạy lại cell này.


In [ ]:
import os

required = [
    f"{DRIVE_PROJECT_DIR}/data/raw/DataTime_export.csv",
    f"{SRC_DIR}/config.py",
    f"{SRC_DIR}/preprocessing.py",
    f"{SRC_DIR}/features.py",
    f"{SRC_DIR}/windowing.py",
    f"{SRC_DIR}/diagnostics.py",
    f"{SRC_DIR}/main_lstm.py",
    f"{SRC_DIR}/main_tcn.py",
    f"{SRC_DIR}/main_seq2seq.py",
    f"{SRC_DIR}/main_lstm_delta.py",
    f"{SRC_DIR}/main_tcn_delta.py",
    f"{SRC_DIR}/main_seq2seq_delta.py",
    f"{SRC_DIR}/models/__init__.py",
    f"{SRC_DIR}/models/lstm.py",
    f"{SRC_DIR}/models/tcn.py",
    f"{SRC_DIR}/models/seq2seq_lstm.py",
]
missing = [p for p in required if not os.path.exists(p)]

if missing:
    print("THIẾU các file sau trên Drive:")
    for p in missing:
        print(" -", p)
    print("\n-> Upload từ máy local (thư mục E:\\sofc\\) vào đúng đường dẫn trên Drive, giữ nguyên cấu trúc thư mục (bao gồm cả src/models/).")
else:
    print("Đã có đủ file cần thiết. Sẵn sàng chạy tiếp.")


## 4. Cài đặt thư viện

`torch`, `pandas`, `scikit-learn` đã có sẵn trên Colab. `xgboost` không cần cho notebook này (RF/XGBoost chạy local) nhưng `models/xgboost_model.py` bị import gián tiếp khi Python quét `models/` package nếu có script nào `import models` trọn gói -- cài luôn cho chắc, không tốn thời gian đáng kể.


In [ ]:
!pip install -q xgboost


## 5. Kiểm tra GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Không có GPU -> Runtime -> Change runtime type -> GPU (T4), rồi Runtime -> Restart session và chạy lại từ đầu.")


## 6. Chạy pipeline (LSTM + TCN + Seq2Seq)

Mỗi model là 1 cell riêng, chạy tuần tự. Mỗi script tự: load + chuẩn bị data (`features.prepare_data()`), chia train/val/test theo run_id, tạo sliding window, train, đánh giá (MAE/RMSE/R2/DTW), rồi lưu model + predictions + bảng kết quả vào `outputs/` -- vì `outputs/` nằm ngay trong `DRIVE_PROJECT_DIR` (đang đứng trên Drive), mọi thứ tự động được giữ lại kể cả khi Colab ngắt kết nối.


In [ ]:
%cd {SRC_DIR}
!python main_lstm.py


In [ ]:
%cd {SRC_DIR}
!python main_tcn.py


In [ ]:
%cd {SRC_DIR}
!python main_seq2seq.py


## 6b. Delta-Target Reformulation (LSTM, TCN, Seq2Seq)

Kết quả raw-target ở mục 6 cho thấy LSTM/TCN thắng đậm ở h=1 nhưng xuống dốc nhanh khi horizon tăng -- dấu hiệu persistence bias (model dựa nhiều vào `V_Lag1`, xem `notes/SOFC_data_notes.md` mục 14.3/16). Delta-Target Reformulation train model trên `y(t+h) - y(t)` thay vì giá trị thô, loại bỏ việc "chép Lag1" như một lối tắt miễn phí.

Baseline raw-target (`main_lstm.py`/`main_tcn.py`/`main_seq2seq.py`, mục 6) **không bị ghi đè** -- 3 script dưới đây ghi kết quả vào `*_delta_results.csv` riêng để so sánh trực tiếp 2 phiên bản.


In [ ]:
%cd {SRC_DIR}
!python main_lstm_delta.py


In [ ]:
%cd {SRC_DIR}
!python main_tcn_delta.py


In [ ]:
%cd {SRC_DIR}
!python main_seq2seq_delta.py


## 7. So sánh 3 model (Colab) — LSTM / TCN / Seq2Seq

Đọc lại `outputs/reports/*.csv` để so sánh, không cần chạy lại cell nào ở mục 6.


In [ ]:
import pandas as pd

REPORTS_DIR = f"{DRIVE_PROJECT_DIR}/outputs/reports"

combined = []
for name, fname in [
    ("LSTM", "lstm_results.csv"), ("TCN", "tcn_results.csv"), ("Seq2Seq", "seq2seq_results.csv"),
    ("LSTM-delta", "lstm_delta_results.csv"), ("TCN-delta", "tcn_delta_results.csv"), ("Seq2Seq-delta", "seq2seq_delta_results.csv"),
]:
    path = f"{REPORTS_DIR}/{fname}"
    if os.path.exists(path):
        d = pd.read_csv(path)
        d.insert(0, "model", name)
        combined.append(d)
    else:
        print(f"(chưa có {fname} -- chạy cell tương ứng ở mục 6 trước)")

combined_df = pd.concat(combined, ignore_index=True) if combined else pd.DataFrame()
combined_df


## 8. Ghi chú

- Kết quả (`outputs/reports/*.csv`), model (`outputs/models_saved/`), và predictions (`outputs/predictions_cache/`) đều nằm trên Drive -> tải về máy local vào đúng `E:\sofc\outputs\` để gộp chung với kết quả Random Forest / XGBoost chạy local (`src/main.py`).
- Random Forest và XGBoost KHÔNG chạy ở đây (không cần GPU, chạy local nhanh hơn nhiều qua `python src/main.py`).
- Hyperparameter (`hidden_size`, `num_channels`, `epochs`, `patience`...) trong `main_lstm.py`/`main_tcn.py`/`main_seq2seq.py` là điểm khởi đầu, chưa tune riêng theo dataset SOFC -- nếu train loss không giảm hoặc R2 âm, sửa trực tiếp trong script đó (giảm `hidden_size`/`num_channels` hoặc tăng `patience`), upload lại lên Drive rồi chạy lại cell tương ứng.
- Muốn train lại 1 model cụ thể: chỉ cần chạy lại đúng cell của model đó ở mục 6, không cần chạy lại toàn bộ notebook.
